# DSPy GEPA prompt optimization (text classification)

This notebook mirrors `03_Advanced_DSPy_Optimizer_GEPA.py` and walks through using the **GEPA** optimizer to improve a smaller Gemini model on a supervised text-classification task. It is adapted from the [Mosaic AI Databricks GEPA post](https://medium.com/@AI-on-Databricks/prompt-optimizing-with-gepa-and-databricks-for-90x-cheaper-inference-0068a2909d86) but stripped of any Databricks dependency.

**Flow**

1. **Dataset setup** — load the PubMed text-classification dataset (previously downloaded to `./dataset/`) and build balanced train/test splits.
2. **DSPy signature + module** — define `TextClassificationSignature` and `TextClassifier` (Gemini via LiteLLM).
3. **Evaluation metric** — exact-match score with feedback strings GEPA can read.
4. **Baseline** — measure uncompiled small and large Gemini accuracy.
5. **GEPA configuration + training** — compile the student using the large Gemini as the reflection LM.
6. **Testing** — re-evaluate the compiled student, print absolute/relative lift, inspect the evolved prompt.

**Prerequisites**

- `GOOGLE_API_KEY` in `.env` at the repo root (auto-loaded by `python-dotenv`).
- Dataset files on disk — run once from the repo root:
  ```bash
  python dspy_hackathon/download_pubmed_dataset.py
  ```
  This creates `dspy_hackathon/dataset/train.csv` and `dspy_hackathon/dataset/test.csv`.

## Install dependencies

Run once (uncomment the cell below) and restart the kernel. `certifi` fixes `SSL: CERTIFICATE_VERIFY_FAILED` on some macOS Python builds.

In [23]:
# %pip install --upgrade dspy mlflow python-dotenv certifi pandas numpy

## 1. Imports and model configuration

- **`small_model`** — the student we want to optimize (cheap / fast).
- **`large_model`** — stronger baseline used as an upper bound.
- **`reflection_model`** — the teacher LM GEPA uses to propose new prompts; defaults to `large_model`.

`load_dotenv()` walks up from the current working directory, so running the notebook from the repo root or from `dspy_hackathon/` both work.

In [24]:
import dataclasses
import io
import json
import os
import ssl
import urllib.request
from pathlib import Path

import certifi
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from dspy.datasets.dataset import Dataset
from pandas import StringDtype

load_dotenv()  # looks in cwd and parents for .env

# Gemini model ids (LiteLLM). Student = smaller; large = stronger baseline; reflection = GEPA teacher LM.
small_model = "gemini/gemini-2.5-flash-lite"
large_model = "gemini/gemini-3.1-pro-preview"
reflection_model = large_model

print(f"small_model      = {small_model}")
print(f"large_model      = {large_model}")
print(f"reflection_model = {reflection_model}")
print(f"GOOGLE_API_KEY set: {bool(os.environ.get('GOOGLE_API_KEY'))}")

small_model      = gemini/gemini-2.5-flash-lite
large_model      = gemini/gemini-3.1-pro-preview
reflection_model = gemini/gemini-3.1-pro-preview
GOOGLE_API_KEY set: True


## 2. Dataset setup

We read the PubMed train/test CSVs from a local `dataset/` folder. If you run this notebook from the repo root, the files live at `./dspy_hackathon/dataset/`; if you launch Jupyter from inside `dspy_hackathon/`, they live at `./dataset/`. The helper below tries both.

**Labels:** `CONCLUSIONS`, `RESULTS`, `METHODS`, `OBJECTIVE`, `BACKGROUND`.

In [25]:
def _to_json_serializable(obj):
    """Recursively convert LM history to JSON-safe data (e.g. LiteLLM ModelResponse objects)."""
    if obj is None or isinstance(obj, (bool, int, float, str)):
        return obj
    if isinstance(obj, dict):
        return {str(k): _to_json_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_json_serializable(x) for x in obj]
    if isinstance(obj, set):
        return [_to_json_serializable(x) for x in sorted(obj, key=str)]
    model_dump = getattr(obj, "model_dump", None)
    if callable(model_dump):
        try:
            return _to_json_serializable(model_dump())
        except Exception:
            pass
    dict_fn = getattr(obj, "dict", None)
    if callable(dict_fn):
        try:
            return _to_json_serializable(dict_fn())
        except Exception:
            pass
    if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
        try:
            return _to_json_serializable(dataclasses.asdict(obj))
        except Exception:
            pass
    return str(obj)


def _read_csv_from_url(url: str) -> pd.DataFrame:
    """Fallback: load a CSV over HTTPS using certifi's CA bundle."""
    print(f"Reading CSV from URL: {url}")
    ctx = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(url, context=ctx, timeout=120) as resp:
        return pd.read_csv(io.BytesIO(resp.read()))


def _read_csv_from_disk(filename: str, dataset_dir: str | None = None) -> pd.DataFrame:
    """Load a CSV from a local dataset directory; checks both `./dataset` and `./dspy_hackathon/dataset`."""
    candidates = [dataset_dir] if dataset_dir else ["./dataset", "./dspy_hackathon/dataset"]
    for d in candidates:
        p = Path(d) / filename
        if p.exists():
            print(f"Reading CSV from disk: {p}")
            return pd.read_csv(p)
    raise FileNotFoundError(
        f"CSV '{filename}' not found in {candidates}. "
        f"Run `python dspy_hackathon/download_pubmed_dataset.py` to download first."
    )


def read_data_and_subset_to_categories() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the PubMed train/test CSVs from disk and drop the unused `description_cln` column."""
    train = _read_csv_from_disk("train.csv")
    test = _read_csv_from_disk("test.csv")
    train.drop("description_cln", axis=1, inplace=True)
    test.drop("description_cln", axis=1, inplace=True)
    return train, test

In [26]:
class CSVDataset(Dataset):
    """Balanced sampler: `n_train_per_label` train + `n_test_per_label` test per class."""

    def __init__(self, n_train_per_label: int = 40, n_test_per_label: int = 20, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.n_train_per_label = n_train_per_label
        self.n_test_per_label = n_test_per_label
        self._create_train_test_split_and_ensure_labels()

    def _create_train_test_split_and_ensure_labels(self) -> None:
        """Perform a train/test split that ensures labels in `test` also appear in `train`."""
        train_df, test_df = read_data_and_subset_to_categories()
        train_df = train_df.astype(StringDtype())
        test_df = test_df.astype(StringDtype())

        train_samples_df = pd.concat(
            [group.sample(n=self.n_train_per_label, random_state=1) for _, group in train_df.groupby("target")]
        )
        test_samples_df = pd.concat(
            [group.sample(n=self.n_test_per_label, random_state=1) for _, group in test_df.groupby("target")]
        )

        self._train = train_samples_df.to_dict(orient="records")
        self._test = test_samples_df.to_dict(orient="records")

In [27]:
# Small splits to keep the demo fast and cheap.
dataset = CSVDataset(n_train_per_label=3, n_test_per_label=10)

train_dataset = [example.with_inputs("description") for example in dataset.train]
test_dataset = [example.with_inputs("description") for example in dataset.test]

print(f"train dataset size: {len(train_dataset)}")
print(f"test dataset size:  {len(test_dataset)}")
print(f"Train labels:       {sorted(set(ex.target for ex in dataset.train))}")

print("\n********* train dataset sample entries (first 5) *********")
for ex in train_dataset[:5]:
    print(ex)
print("\n********* test dataset sample entries (first 5) *********")
for ex in test_dataset[:5]:
    print(ex)

Reading CSV from disk: dataset/train.csv
Reading CSV from disk: dataset/test.csv
train dataset size: 15
test dataset size:  50
Train labels:       ['BACKGROUND', 'CONCLUSIONS', 'METHODS', 'OBJECTIVE', 'RESULTS']

********* train dataset sample entries (first 5) *********
Example({'target': 'BACKGROUND', 'description': 'Although opioids are effective treatments for postoperative pain , they contribute to the delayed recovery of gastrointestinal function .'}) (input_keys={'description'})
Example({'target': 'OBJECTIVE', 'description': 'This study evaluated whether increased exercise in hospital and afterward would shorten length of stay and improve physical function at 1 month .'}) (input_keys={'description'})
Example({'target': 'OBJECTIVE', 'description': 'There is a need to identify effective interventions to promote walking capacity in seniors .'}) (input_keys={'description'})
Example({'target': 'CONCLUSIONS', 'description': 'In the brief period since introduction of dihydroartemisinin

## 3. DSPy signature and classifier module

- **`TextClassificationSignature`** — typed I/O contract for one LM call. `target` is constrained to the 5 label strings (structured output).
- **`TextClassifier`** — a `dspy.Module` that wraps `dspy.Predict(TextClassificationSignature)` and pins the chosen Gemini model.

MLflow autologging is turned on so compile/eval traces are captured if you browse `mlflow ui`.

In [28]:
from typing import Literal
import warnings

import dspy

# Opik (imported transitively by mlflow) warns about Pydantic v1 on Python 3.14+; silence it.
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=r".*Pydantic V1 functionality isn't compatible with Python 3\.\d+.*",
)
import mlflow


def gemini_llm(model: str, cache: bool = False, **kwargs) -> dspy.LM:
    return dspy.LM(model, api_key=os.environ["GOOGLE_API_KEY"], cache=cache, **kwargs)


mlflow.dspy.autolog(log_evals=True, log_compiles=True, log_traces_from_compile=True)

In [29]:
class TextClassificationSignature(dspy.Signature):
    description: str = dspy.InputField()
    target: Literal["CONCLUSIONS", "RESULTS", "METHODS", "OBJECTIVE", "BACKGROUND"] = dspy.OutputField()


class TextClassifier(dspy.Module):
    """Classifies medical-text fragments into one of five PubMed structural categories."""

    def __init__(self, model: str):
        super().__init__()
        self.lm = gemini_llm(model, cache=False, max_tokens=25000)
        self.generate_classification = dspy.Predict(TextClassificationSignature)

    def forward(self, description: str):
        with dspy.context(lm=self.lm):
            return self.generate_classification(description=description)

In [30]:
# Smoke test: run the student on one made-up description.
text_classifier = TextClassifier(model=small_model)
description = (
    "This study is designed as a randomised controlled trial in which men living with HIV in Australia "
    "will be assigned to either an intervention group or usual care control group ."
)
print(f"Using {small_model} to classify:\n{description}\n")
print(text_classifier(description=description))



Using gemini/gemini-2.5-flash-lite to classify:
This study is designed as a randomised controlled trial in which men living with HIV in Australia will be assigned to either an intervention group or usual care control group .

Prediction(
    target='METHODS'
)


In [31]:
# now run the smoke test on the large model
text_classifier_large = TextClassifier(model=large_model)
print(f"Using {large_model} to classify:\n{description}\n")
print(text_classifier_large(description=description))

Using gemini/gemini-3.1-pro-preview to classify:
This study is designed as a randomised controlled trial in which men living with HIV in Australia will be assigned to either an intervention group or usual care control group .

Prediction(
    target='METHODS'
)


## 4. Evaluation metric

GEPA expects a metric that returns a `dspy.Prediction` with a numeric `score` and a text `feedback`. Feedback is the signal GEPA's reflection LM reads when it proposes prompt rewrites.

In [32]:
from tqdm.auto import tqdm


def validate_classification_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    """Score 1 if predicted label matches gold, else 0; feedback string helps GEPA refine prompts."""
    if example.target == prediction.target:
        return dspy.Prediction(score=1, feedback="Correct: prediction matches the gold label.")
    return dspy.Prediction(
        score=0,
        feedback=f"Incorrect: expected '{example.target}' but model predicted '{prediction.target}'.",
    )


def check_accuracy_on_test_dataset(classifier, test_data=test_dataset, desc: str = "Evaluating") -> float:
    """Run the classifier across `test_data` and return mean exact-match accuracy."""
    scores = []
    progress = tqdm(test_data, desc=desc, unit="ex")
    for example in progress:
        prediction = classifier(description=example["description"])
        scores.append(validate_classification_with_feedback(example, prediction).score)
        progress.set_postfix(acc=f"{np.mean(scores):.3f}")
    return float(np.mean(scores))

## 5. Baseline: small vs. large Gemini (before GEPA)

Before optimizing, we capture:

- The **original prompt** DSPy generates from `TextClassificationSignature` for the student (by running one forward pass and peeking at `lm.history`).
- The **uncompiled student accuracy** and the **large-model accuracy** on the 50-example test set (our ceiling reference).

In [33]:
baseline_classifier = TextClassifier(model=small_model)
_ = baseline_classifier(description=description)
original_prompt = baseline_classifier.lm.history[-1]["messages"][0]["content"]
print(f"=== Original (pre-GEPA) prompt for {small_model} ===\n")
print(original_prompt)

=== Original (pre-GEPA) prompt for gemini/gemini-2.5-flash-lite ===

Your input fields are:
1. `description` (str):
Your output fields are:
1. `target` (Literal['CONCLUSIONS', 'RESULTS', 'METHODS', 'OBJECTIVE', 'BACKGROUND']):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## description ## ]]
{description}

[[ ## target ## ]]
{target}        # note: the value you produce must exactly match (no extra characters) one of: CONCLUSIONS; RESULTS; METHODS; OBJECTIVE; BACKGROUND

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `description`, produce the fields `target`.


In [34]:
uncompiled_small_lm_accuracy = check_accuracy_on_test_dataset(
    TextClassifier(model=small_model), desc=f"Uncompiled {small_model}"
)
print(f"Uncompiled {small_model} accuracy on test dataset: {uncompiled_small_lm_accuracy:.4f}")


uncompiled_large_lm_accuracy = check_accuracy_on_test_dataset(
    TextClassifier(model=large_model), desc=f"Uncompiled {large_model}"
)
print(f"Uncompiled {large_model} accuracy on test dataset: {uncompiled_large_lm_accuracy:.4f}")

Uncompiled gemini/gemini-2.5-flash-lite accuracy on test dataset: 0.6800


KeyboardInterrupt: 

## 6. GEPA configuration

**What each knob does**

- **`metric=`** — `validate_classification_with_feedback` above; returns `score + feedback`.
- **`auto="light"`** — pre-set budget; DSPy will print something like *"Running GEPA for approx 440 metric calls / 29.33 full evals on the train set."*
- **`reflection_minibatch_size=15`** — how many examples GEPA shows to the reflection LM when proposing a new prompt. We match the trainset size.
- **`reflection_lm=`** — the larger Gemini (`reflection_model`). This is the "teacher" that rewrites instructions.
- **`num_threads=16`, `seed=1`** — parallelism and reproducibility.

*Note:* we pass only `trainset` (no `valset`), so GEPA will log `No valset provided; using trainset as valset.` For a real run, split a held-out `valset` and pass it explicitly.

In [ ]:
import uuid

run_id = str(uuid.uuid4())
print(f"run_id: {run_id}")

gepa = dspy.GEPA(
    metric=validate_classification_with_feedback,
    auto="light",
    reflection_minibatch_size=15,
    reflection_lm=gemini_llm(reflection_model, max_tokens=100000),
    num_threads=16,
    seed=1,
)

## 7. Training: run GEPA (compile)

`gepa.compile(...)` runs the optimization loop: evaluate candidates on the trainset, let the reflection LM propose prompt rewrites, keep the Pareto front, repeat until the budget is exhausted. The resulting program is saved to JSON so we can reload it later.

This cell does real API traffic to both `small_model` (many times) and `reflection_model` (a handful of times). Expect minutes, not seconds.

In [ ]:
from tqdm.auto import tqdm

print("Starting GEPA optimization...")

# Wrap GEPA's metric so we can drive a tqdm bar from each scoring call.
# `max_metric_calls` is the budget DSPy computes from `auto="light"`.
_orig_metric = gepa.metric_fn
_progress = tqdm(total=gepa.max_metric_calls, desc="GEPA metric calls", unit="call")
_correct = 0


def _metric_with_progress(example, prediction, trace=None, pred_name=None, pred_trace=None):
    global _correct
    result = _orig_metric(example, prediction, trace, pred_name, pred_trace)
    _correct += int(getattr(result, "score", 0) or 0)
    _progress.update(1)
    _progress.set_postfix(acc=f"{_correct / _progress.n:.3f}")
    return result


gepa.metric_fn = _metric_with_progress

try:
    with mlflow.start_run(run_name=f"gepa_{run_id}"):
        compiled_gepa = gepa.compile(
            TextClassifier(model=small_model),
            trainset=train_dataset,  # reminder: only 15 examples here
        )
finally:
    _progress.close()
    gepa.metric_fn = _orig_metric

compiled_path = f"compiled_gepa_{run_id}.json"
compiled_gepa.save(compiled_path)
print(f"GEPA optimization completed. Saved: {compiled_path}")

## 8. Testing: re-evaluate the compiled student

Reload the optimized program into a fresh `TextClassifier` bound to the **same small Gemini** and measure accuracy on the held-out test set. We then compute absolute and relative lift vs. the uncompiled baseline.

In [ ]:
print("Loading the optimized program and re-evaluating on the test dataset...")
text_classifier_gepa = TextClassifier(model=small_model)
text_classifier_gepa.load(compiled_path)

compiled_small_lm_accuracy = check_accuracy_on_test_dataset(
    text_classifier_gepa, desc=f"GEPA-compiled {small_model}"
)
print(f"Compiled {small_model} accuracy on test dataset: {compiled_small_lm_accuracy:.4f}")

abs_gain = compiled_small_lm_accuracy - uncompiled_small_lm_accuracy
rel_pct = (abs_gain / uncompiled_small_lm_accuracy) * 100.0 if uncompiled_small_lm_accuracy else float("nan")
print(
    f"GEPA vs uncompiled student ({small_model}): "
    f"accuracy {uncompiled_small_lm_accuracy:.4f} \u2192 {compiled_small_lm_accuracy:.4f} "
    f"(\u0394 = {abs_gain:+.4f}; {rel_pct:+.2f}% relative vs. uncompiled baseline)"
)
print(f"Large-model ceiling on same test set: {uncompiled_large_lm_accuracy:.4f}")

## 9. Inspect the evolved prompt and dump history

The compiled JSON contains the rewritten signature instructions. The cell below also prints the exact system/message sent to the student on its most recent call and dumps the full LM history to `gepa_<run_id>_history.json` (after converting LiteLLM `ModelResponse` objects to JSON-safe dicts).

In [ ]:
print("=== Evolved (post-GEPA) prompt sent to the student ===\n")
print(text_classifier_gepa.lm.history[-1]["messages"][0]["content"])

history_path = f"gepa_{run_id}_history.json"
with open(history_path, "w") as f:
    json.dump(_to_json_serializable(text_classifier_gepa.lm.history), f, indent=2)
    f.write("\n")
print(f"\nHistory dumped to {history_path}")